# 05 — Modelo Global LightGBM

**Dataset:** `receita_realizada_a_partir_2019_sgo.parquet`  
**Série-alvo:** Fonte Detalhada Harmonizada (`Fonte_Det_Cód_Harm`)  
**Horizonte de previsão:** 12 meses (nov/2025 – out/2026)

**Objetivo:** Comparar o modelo global LightGBM contra os modelos individuais
(ARIMA/Ridge/SVR/MLP por série) e, quando disponíveis, os modelos pooled por cluster.

## Estratégia: Modelo Global

Um único LightGBM é treinado simultaneamente em **todas as séries não-ruído-branco**.
A identidade de cada série (`series_id`) é passada como feature categórica nativa do LightGBM,
permitindo ao modelo aprender tanto padrões universais (sazonalidade fiscal, tendência)
quanto idiossincrasias de cada fonte de receita.

## Features do modelo
| Feature | Tipo | Descrição |
|---|---|---|
| `lag_1` … `lag_12` | Numérica | Valores z-score dos 12 meses anteriores |
| `month_sin`, `month_cos` | Numérica | Mês em codificação cíclica (continuidade dez→jan) |
| `quarter` | Numérica | Trimestre fiscal 1–4 |
| `year_offset` | Numérica | Ano − 2019 (proxy de tendência secular) |
| `series_id` | **Categórica** | Código da fonte (nativo LightGBM) |

## Referências
- **Ke et al. (2017)** — *LightGBM: A High Performance Gradient Boosting Framework.* NeurIPS.
- **Makridakis et al. (2022)** — *The M5 Accuracy Competition.* IJF, v.38, n.4. (LightGBM dominou.)
- **Montero-Manso et al. (2020)** — *FFORMA: Feature-based Forecast Model Averaging.* IJF, v.36, n.1.
- **Bergmeir & Benítez (2018)** — *Cross-validation for TS prediction.* CSDA, v.120.
- **Hyndman & Athanasopoulos (2021)** — *Forecasting: Principles and Practice*, 3.ed., OTexts.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

from experiment_lgbm import rodar_experimento_lgbm

CORES = {
    'azul'    : '#1B3A5C',
    'azul_m'  : '#2E6DA4',
    'verde'   : '#27AE60',
    'vermelho': '#C0392B',
    'amarelo' : '#F39C12',
    'roxo'    : '#8E44AD',
    'laranja' : '#E67E22',
    'cinza'   : '#BDC3C7',
}

plt.rcParams.update({
    'figure.dpi': 130, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('Ambiente pronto.')

## 0 · Execução do experimento

In [ ]:
# Executa o experimento global LightGBM
# Salva automaticamente resultados em results/comparacao_lgbm_*.csv e .xlsx
resultados = rodar_experimento_lgbm(verbose=True)

comp     = resultados['comparacao_df']
modelo   = resultados['modelo']
n_est    = resultados['n_estimators']
feat_imp = resultados['feature_imp']
n_treino = resultados['n_series_treino']

print(f"\nSeries avaliadas  : {len(comp)}")
print(f"Series no treino  : {n_treino}")
print(f"n_estimators      : {n_est}")
print(f"Wilcoxon p-valor  : {resultados['wilcoxon_p']:.4f}")
comp.head()

## 1 · Importância de Features (gain)

In [ ]:
top_n = min(20, len(feat_imp))
df_fi = feat_imp.head(top_n).copy()

def _cor_feat(name):
    if name.startswith('lag_') : return CORES['azul_m']
    if name == 'series_id'     : return CORES['roxo']
    return CORES['verde']

cores_fi = [_cor_feat(f) for f in df_fi['feature']]

fig, ax = plt.subplots(figsize=(9, max(4, top_n * 0.42)))
ax.barh(range(top_n), df_fi['importance'], color=cores_fi,
        edgecolor='white', height=0.75)
ax.set_yticks(range(top_n))
ax.set_yticklabels(df_fi['feature'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Importância acumulada (gain)', fontsize=10)
ax.set_title(f'Top {top_n} features — LightGBM Global', fontsize=11)

patches = [
    mpatches.Patch(color=CORES['azul_m'], label='Lag features (lags 1–12)'),
    mpatches.Patch(color=CORES['verde'],  label='Calendário (mês, trimestre, ano)'),
    mpatches.Patch(color=CORES['roxo'],   label='series_id (identidade da série)'),
]
ax.legend(handles=patches, fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

# Importância agrupada por tipo
df_fi['tipo'] = df_fi['feature'].apply(
    lambda x: 'Lag' if x.startswith('lag_') else ('Series ID' if x == 'series_id' else 'Calendario')
)
por_tipo = df_fi.groupby('tipo')['importance'].sum()
total    = por_tipo.sum()
print("\nImportancia por grupo de features:")
for tipo, val in por_tipo.sort_values(ascending=False).items():
    print(f"  {tipo:<15}: {val:>12,.0f}  ({100*val/total:.1f}%)")

## 2 · Comparação de RMSE: Individual vs LightGBM

In [ ]:
import glob as _glob

cols_cmp  = ['Ind_RMSE', 'LGBM_RMSE']
labels_m  = ['Individual\n(melhor)', 'LightGBM\nGlobal']
cores_m   = [CORES['azul'], CORES['verde']]

# Tenta incluir Pooled se disponível
local_csvs = sorted(_glob.glob('../results/comparacao_local_*.csv'))
df_local   = None
if local_csvs:
    df_local = pd.read_csv(local_csvs[-1])
    df_local['Codigo'] = df_local['Codigo'].astype(str)

comp_plot = comp[['Codigo'] + [c for c in cols_cmp if c in comp.columns]].copy()
comp_plot['Codigo'] = comp_plot['Codigo'].astype(str)

if df_local is not None and 'Pooled_Melhor_RMSE' in df_local.columns:
    comp_plot = comp_plot.merge(df_local[['Codigo','Pooled_Melhor_RMSE']], on='Codigo', how='left')
    cols_cmp  = ['Ind_RMSE', 'Pooled_Melhor_RMSE', 'LGBM_RMSE']
    labels_m  = ['Individual\n(melhor)', 'Pooled\n(melhor)', 'LightGBM\nGlobal']
    cores_m   = [CORES['azul'], CORES['laranja'], CORES['verde']]

cols_cmp = [c for c in cols_cmp if c in comp_plot.columns]
cv       = comp_plot.dropna(subset=cols_cmp)
medias   = [cv[c].mean()   for c in cols_cmp]
medianas = [cv[c].median() for c in cols_cmp]
x        = np.arange(len(cols_cmp))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, vals, titulo in zip(axes, [medias, medianas], ['Media', 'Mediana']):
    bars = ax.bar(x, vals, width=0.6, color=cores_m, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                v + max(vals)*0.01, f'R${v/1e6:.2f}M',
                ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_m, fontsize=9)
    ax.set_title(f'RMSE {titulo}  (n={len(cv)})', fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R${v/1e6:.1f}M'))

plt.suptitle('Comparacao de RMSE por abordagem de modelagem', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3 · Individual vs LightGBM — Dumbbell por série

In [ ]:
if 'Ind_RMSE' in comp.columns:
    df_d = comp.dropna(subset=['Ind_RMSE','LGBM_RMSE']).copy()
    df_d = df_d.sort_values('Ind_RMSE', ascending=False).reset_index(drop=True)
    df_d['ganhou'] = df_d['LGBM_RMSE'] < df_d['Ind_RMSE']

    n = len(df_d)
    y = np.arange(n)
    fig, ax = plt.subplots(figsize=(10, max(5, n * 0.42)))

    for i, row in df_d.iterrows():
        cor = CORES['verde'] if row['ganhou'] else CORES['vermelho']
        ax.plot([row['Ind_RMSE'], row['LGBM_RMSE']], [i, i],
                color=cor, lw=1.5, alpha=0.7, zorder=1)

    ax.scatter(df_d['Ind_RMSE'], y, color=CORES['azul'], s=70, zorder=3,
               label='Individual')
    ax.scatter(df_d['LGBM_RMSE'], y,
               color=[CORES['verde'] if g else CORES['vermelho'] for g in df_d['ganhou']],
               marker='D', s=70, zorder=3, label='LightGBM')

    ax.set_yticks(y)
    ax.set_yticklabels(df_d['Codigo'].astype(str), fontsize=8)
    ax.set_xlabel('RMSE (holdout)', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))

    n_g = int(df_d['ganhou'].sum())
    ax.set_title(f'Individual vs LightGBM Global | LGBM melhorou em {n_g}/{n}', fontsize=11)

    ax.legend(handles=[
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=CORES['azul'],   markersize=8, label='Individual'),
        plt.Line2D([0],[0], marker='D', color='w', markerfacecolor=CORES['azul_m'], markersize=8, label='LightGBM'),
        mpatches.Patch(color=CORES['verde'],   label='LGBM melhor'),
        mpatches.Patch(color=CORES['vermelho'], label='Individual melhor'),
    ], fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f'LightGBM melhor em {n_g}/{n} series ({100*n_g/n:.1f}%)')

## 4 · Delta percentual por série

In [ ]:
if 'Delta_RMSE_pct' in comp.columns:
    df_delta = comp.dropna(subset=['Delta_RMSE_pct']).sort_values('Delta_RMSE_pct')
    n_d      = len(df_delta)

    fig, ax = plt.subplots(figsize=(9, max(4, n_d * 0.40)))
    cores_bar = [CORES['verde'] if v < 0 else CORES['vermelho']
                 for v in df_delta['Delta_RMSE_pct']]
    bars = ax.barh(range(n_d), df_delta['Delta_RMSE_pct'],
                   color=cores_bar, edgecolor='white', height=0.7)

    for i, (bar, v) in enumerate(zip(bars, df_delta['Delta_RMSE_pct'])):
        ha = 'right' if v < 0 else 'left'
        ax.text(v + (-2 if v < 0 else 2), bar.get_y() + bar.get_height()/2,
                f'{v:+.1f}%', ha=ha, va='center', fontsize=8)

    ax.axvline(0, color='black', lw=1.2)
    ax.set_yticks(range(n_d))
    ax.set_yticklabels(df_delta['Codigo'].astype(str), fontsize=8)
    ax.set_xlabel('Delta RMSE LGBM vs Individual (%)', fontsize=10)

    n_mel = int((df_delta['Delta_RMSE_pct'] < 0).sum())
    ax.set_title(f'Delta % RMSE | negativo = LGBM melhorou | {n_mel}/{n_d} series', fontsize=11)
    plt.tight_layout()
    plt.show()

## 5 · Teste de Wilcoxon Signed-Rank

In [ ]:
stat_w = resultados.get('wilcoxon_stat', float('nan'))
p_w    = resultados.get('wilcoxon_p',    float('nan'))

print('=' * 65)
print('TESTE DE WILCOXON SIGNED-RANK')
print('H0: RMSE_individual = RMSE_lgbm  (mediana das diferencas = 0)')
print('H1: RMSE_individual != RMSE_lgbm (bicaudal)')
print('=' * 65)
if not (np.isnan(stat_w) or np.isnan(p_w)):
    print(f'Estatistica W : {stat_w:.2f}')
    print(f'p-valor       : {p_w:.4f}')
    if   p_w < 0.05: print('=> Rejeita H0 a 5%: diferenca SIGNIFICATIVA')
    elif p_w < 0.10: print('=> Rejeita H0 a 10%: diferenca MARGINALMENTE significativa')
    else:            print('=> Nao rejeita H0 a 5%: diferenca NAO significativa')
else:
    print('Amostras insuficientes para o teste (n < 5).')

print()
print('RESUMO QUANTITATIVO')
print('-' * 65)
print(f'Series avaliadas          : {len(comp)}')
print(f'n_estimators (opt.)       : {n_est}')
print(f'Features totais           : {len(feat_imp)}')

if 'Ind_RMSE' in comp.columns:
    n_mel = int((comp['LGBM_RMSE'] < comp['Ind_RMSE']).sum())
    print(f'LGBM melhor que individual : {n_mel}/{len(comp)} series')
    print(f'RMSE medio individual      : R$ {comp["Ind_RMSE"].mean():>14,.2f}')
    print(f'RMSE medio LGBM            : R$ {comp["LGBM_RMSE"].mean():>14,.2f}')
    print(f'RMSE mediano individual    : R$ {comp["Ind_RMSE"].median():>14,.2f}')
    print(f'RMSE mediano LGBM          : R$ {comp["LGBM_RMSE"].median():>14,.2f}')
    if 'Delta_RMSE_pct' in comp.columns:
        print(f'Delta medio                : {comp["Delta_RMSE_pct"].mean():+.2f}%')
        print(f'Delta mediano              : {comp["Delta_RMSE_pct"].median():+.2f}%')

## 6 · Comparação tripla: Individual × Pooled × LightGBM

Executa somente se os resultados do `experiment_local.py` estiverem disponíveis.

In [ ]:
import glob as _glob

local_csvs = sorted(_glob.glob('../results/comparacao_local_*.csv'))
if local_csvs:
    df_loc = pd.read_csv(local_csvs[-1])
    df_loc['Codigo'] = df_loc['Codigo'].astype(str)
    comp['Codigo']   = comp['Codigo'].astype(str)

    df_3 = comp.merge(
        df_loc[['Codigo','Pooled_Melhor_RMSE','Pooled_Melhor_Modelo']],
        on='Codigo', how='inner'
    ).dropna(subset=['Ind_RMSE','Pooled_Melhor_RMSE','LGBM_RMSE'])

    n3 = len(df_3)
    print(f'{n3} series com os tres modelos disponiveis')

    df_3['vencedor'] = df_3[['Ind_RMSE','Pooled_Melhor_RMSE','LGBM_RMSE']].idxmin(axis=1)
    contagem = df_3['vencedor'].value_counts()

    cols3  = ['Ind_RMSE','Pooled_Melhor_RMSE','LGBM_RMSE']
    labs3  = ['Individual','Pooled','LightGBM']
    cores3 = [CORES['azul'], CORES['laranja'], CORES['verde']]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Barras de RMSE medio
    meds = [df_3[c].mean() for c in cols3]
    bars = axes[0].bar(range(3), meds, color=cores3, edgecolor='white', width=0.6)
    for bar, v in zip(bars, meds):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     v + max(meds)*0.01, f'R${v/1e6:.2f}M',
                     ha='center', va='bottom', fontsize=8)
    axes[0].set_xticks(range(3))
    axes[0].set_xticklabels(labs3, fontsize=10)
    axes[0].set_title(f'RMSE medio (n={n3})', fontsize=11)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R${v/1e6:.1f}M'))

    # Pizza de vencedores
    nome_map  = dict(zip(cols3, labs3))
    win_vals  = [contagem.get(c, 0) for c in cols3]
    win_labs  = [f'{nome_map[c]}\n({contagem.get(c,0)} series)' for c in cols3]
    axes[1].pie(win_vals, labels=win_labs, colors=cores3,
                autopct='%1.0f%%', startangle=90, textprops={'fontsize': 9})
    axes[1].set_title('Vencedor por serie (menor RMSE)', fontsize=11)

    plt.suptitle('Comparacao Tripla: Individual x Pooled x LightGBM Global',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\nVencedores por abordagem (n={n3} series):")
    for c in cols3:
        n_w = contagem.get(c, 0)
        print(f"  {nome_map[c]:<15}: {n_w:3d} series ({100*n_w/n3:.1f}%)")
else:
    print('Resultados do experimento local ainda nao disponiveis.')
    print('Execute experiment_local.py e retorne a este notebook.')

## 7 · Top melhorias e pioras

In [ ]:
if 'Delta_RMSE_pct' in comp.columns and 'Ind_RMSE' in comp.columns:
    cols_show = ['Codigo', 'Ind_Melhor_Modelo', 'N_Treino', 'Ind_RMSE', 'LGBM_RMSE', 'Delta_RMSE_pct']
    cols_show = [c for c in cols_show if c in comp.columns]

    top5 = comp[comp['Delta_RMSE_pct'] < 0].nsmallest(5, 'Delta_RMSE_pct')[cols_show]
    bot5 = comp[comp['Delta_RMSE_pct'] > 0].nlargest(5,  'Delta_RMSE_pct')[cols_show]

    print('TOP 5 — LGBM mais melhorou:')
    print(top5.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))
    print()
    print('TOP 5 — LGBM mais piorou:')
    print(bot5.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))

## 8 · Tabela completa

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

cols_show = [
    'Codigo', 'N_Treino', 'N_Holdout',
    'Ind_Melhor_Modelo', 'Ind_RMSE',
    'LGBM_RMSE', 'LGBM_MAE', 'LGBM_MAPE',
    'Delta_RMSE_pct',
]
cols_show = [c for c in cols_show if c in comp.columns]

def _hl(row):
    styles = [''] * len(row)
    if 'Delta_RMSE_pct' in row.index:
        idx = list(row.index).index('Delta_RMSE_pct')
        val = row['Delta_RMSE_pct']
        if not pd.isna(val):
            styles[idx] = ('background-color:#D5F5E3;color:#1E8449'
                           if val < 0 else 'background-color:#FDECEA;color:#C0392B')
    return styles

fmt = {c: '{:,.0f}' for c in cols_show if 'RMSE' in c or 'MAE' in c}
if 'Delta_RMSE_pct' in cols_show: fmt['Delta_RMSE_pct'] = '{:+.1f}%'
if 'LGBM_MAPE'      in cols_show: fmt['LGBM_MAPE']      = '{:.2f}%'

(
    comp[cols_show].style
    .apply(_hl, axis=1)
    .format(fmt, na_rep='N/A')
    .set_caption('LightGBM Global vs Individual | delta negativo = LGBM melhorou')
)

## 9 · Síntese e interpretação

### Por que usar LightGBM para previsão de receitas fiscais?

| Critério | Justificativa |
|---|---|
| **Competições (M4/M5)** | Gradient boosting dominou as competições de previsão mais importantes (2018–2022) |
| **Dados tabulares** | Receitas fiscais são representadas por lags + calendário — domínio natural de árvores |
| **Cross-learning** | O modelo aprende padrões compartilhados entre fontes correlacionadas |
| **Séries curtas** | Séries com poucos dados se beneficiam do sinal das demais |
| **Categóricas nativas** | `series_id` é tratado nativamente sem one-hot encoding |
| **Velocidade** | Treinamento em segundos; ARIMA por série + grids levam minutos/horas |
| **Interpretabilidade** | Feature importance (gain) revela quais lags e padrões de calendário dominam |

### Quando o LightGBM supera os modelos individuais
- Séries com **poucos dados de treino** (≤ 20 obs.) onde modelos individuais sofrem overfitting
- Séries **sem sazonalidade forte** onde ARIMA não tem vantagem estrutural
- Fontes com comportamento correlacionado a outras do mesmo grupo orçamentário
- Séries onde o modelo individual selecionado é sub-ótimo (ex.: Ridge com sinal fraco)

### Quando o modelo individual ainda é melhor
- Séries com **sazonalidade anual muito forte** capturada precisamente pelo ARIMA
- Séries **outliers extremos** que "contaminam" o sinal das demais no treino global
- Séries com **comportamento estrutural único** que o `series_id` não consegue capturar sozinho

### Extensões futuras
1. **Ensembling**: combinar previsão global + individual por série (blend por RMSE no holdout)
2. **Optuna**: tuning bayesiano dos hiperparâmetros (learning_rate, num_leaves, etc.)
3. **Reconciliação hierárquica**: agregar previsões ao nível de Origem/Espécie (método MinT)
4. **Temporal Fusion Transformer (TFT)**: explorar atenção temporal multi-série
5. **Features de covariáveis**: IPCA, PIB, variáveis macroeconômicas como features extras